# Workshop: Governance & Security

> *"Before go-live: secure RetailHub data with Column Masks, Row Filters, GRANT/REVOKE, and INFORMATION_SCHEMA."*

## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
# Recreate Bronze tables from dataset files (in case they don't exist)
for tbl, fmt, path in [
    ("customers", "csv", f"{DATASET_PATH}/customers/customers.csv"),
    ("orders", "json", f"{DATASET_PATH}/orders/orders_batch.json"),
    ("products", "csv", f"{DATASET_PATH}/products/products.csv"),
]:
    full_name = f"{CATALOG}.{BRONZE_SCHEMA}.{tbl}"
    if not spark.catalog.tableExists(full_name):
        reader = spark.read.format(fmt)
        if fmt == "csv":
            reader = reader.option("header", "true").option("inferSchema", "true")
        reader.load(path).write.mode("overwrite").saveAsTable(full_name)
        print(f"[RECREATED] {full_name}")
    else:
        print(f"[OK] {full_name} exists")

In [0]:
# Ensure Silver tables exist (create from Bronze if missing)
silver_customers = f"{CATALOG}.{SILVER_SCHEMA}.customers"
silver_orders = f"{CATALOG}.{SILVER_SCHEMA}.orders"

if not spark.catalog.tableExists(silver_customers):
    spark.sql(f"""
        CREATE TABLE {silver_customers} AS
        SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    """)
    print(f"Created {silver_customers} from Bronze")

if not spark.catalog.tableExists(silver_orders):
    spark.sql(f"""
        CREATE TABLE {silver_orders} AS
        SELECT *,
            CASE 
                WHEN CAST(REGEXP_EXTRACT(store_id, '(\\\\d+)') AS INT) <= 10 THEN 'East'
                WHEN CAST(REGEXP_EXTRACT(store_id, '(\\\\d+)') AS INT) <= 20 THEN 'West'
                ELSE 'Central'
            END AS store_region
        FROM {CATALOG}.{BRONZE_SCHEMA}.orders
    """)
    print(f"Created {silver_orders} from Bronze (with store_region)")

df_customers = spark.table(silver_customers)
df_orders = spark.table(silver_orders)
print(f"Customers: {df_customers.count()}, Orders: {df_orders.count()}")

## Task 1: Grant Privileges

Grant `SELECT` on the Silver schema to the `analysts` group.

Hint: You need `USE CATALOG`, `USE SCHEMA`, and `SELECT` grants.

**Guidance — Task 1**

Unity Catalog access requires **three stacked grants**. Missing any one of them blocks access entirely.

```sql
-- Step 1: allow the principal to enter the catalog
GRANT USE CATALOG ON CATALOG my_catalog TO `analysts`;

-- Step 2: allow the principal to enter the schema
GRANT USE SCHEMA ON SCHEMA my_catalog.silver TO `analysts`;

-- Step 3: allow the principal to read tables
GRANT SELECT ON SCHEMA my_catalog.silver TO `analysts`;
```

`SELECT ON SCHEMA` grants read access to **all current and future tables** in that schema — no need to grant per table.

| Privilege | Object level | Effect |
|-----------|-------------|--------|
| `USE CATALOG` | Catalog | Required to navigate into the catalog |
| `USE SCHEMA` | Schema | Required to navigate into the schema |
| `SELECT` | Schema or Table | Allows reading data |

**Things to think about**
- What happens if you grant `SELECT` on the schema but forget `USE CATALOG`?
- How would you grant access to a single table instead of the whole schema?

In [0]:
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `analysts`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`")
spark.sql(f"GRANT SELECT ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`")

In [0]:
# Verification
grants_df = spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG}.{SILVER_SCHEMA}")
grants_df.display()

assert grants_df.filter("Principal = 'analysts'").count() >= 2, "Grants not found for analysts group"
print("Task 1 PASSED")

## Task 2: Query INFORMATION_SCHEMA

List all tables in your catalog using `INFORMATION_SCHEMA.TABLES`.

Hint: Filter by `table_schema` to see only Silver tables.

**Guidance — Task 2**

`INFORMATION_SCHEMA` is a set of read-only views built into every Unity Catalog. It follows the SQL standard and exposes catalog metadata without any special tools.

**Syntax:**
```sql
SELECT table_schema, table_name, table_type
FROM my_catalog.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'silver'
```

Use the **three-part name** `{CATALOG}.INFORMATION_SCHEMA.TABLES` to scope the query to your catalog.

**Key INFORMATION_SCHEMA views:**

| View | What it shows |
|------|--------------|
| `TABLES` | All tables and views in the catalog |
| `COLUMNS` | Column-level metadata (name, type, position) |
| `TABLE_PRIVILEGES` | Granted privileges on tables (used in Task 7) |

**Things to think about**
- What columns does `INFORMATION_SCHEMA.COLUMNS` expose?
- How is `INFORMATION_SCHEMA` different from running `DESCRIBE TABLE`?

In [0]:
tables_df = spark.sql(f"""
    SELECT table_schema, table_name, table_type
    FROM {CATALOG}.INFORMATION_SCHEMA.TABLES
    WHERE table_schema = '{SILVER_SCHEMA}'
""")
tables_df.display()

In [0]:
# Verification
assert tables_df.count() > 0, "No tables found in Silver schema"
print(f"Found {tables_df.count()} tables in Silver schema")
print("Task 2 PASSED")

## Task 3: Create a Column Mask Function

Create a SQL function that masks email addresses. Non-admin users should see only the first 2 characters + `***@***.***`.

Hint: Use `is_account_group_member('admins')` and `CONCAT(LEFT(email, 2), '***@***.***')`.

**Guidance — Task 3**

A **Column Mask** is a SQL function attached to a column. When a user queries that column, the function runs automatically and returns either the real value or a masked version based on the caller's identity.

**Syntax:**
```sql
CREATE OR REPLACE FUNCTION catalog.schema.mask_email(email STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('admins') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@***.***')
END
```

Key points:
- The **argument type must match the column type** it will be applied to (STRING for email)
- `is_account_group_member('group_name')` returns TRUE if the calling user is in that group
- Admins see the full value; all other users see the masked version
- The masking is **transparent** — users do not know the original value exists

**Things to think about**
- The function is created inside `SILVER_SCHEMA` — why does the placement in the catalog matter?
- Could you write a mask that returns different masked formats for different groups?

In [0]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.mask_email(email STRING)
    RETURNS STRING
    RETURN CASE
        WHEN is_account_group_member('admins') THEN email
        ELSE CONCAT(LEFT(email, 2), '***@***.***')
    END
""")
print("Masking function created")

In [0]:
# Verification -- test the function
test_df = spark.sql(f"SELECT {CATALOG}.{SILVER_SCHEMA}.mask_email('john.doe@example.com') AS masked")
test_df.display()
print("Task 3 PASSED")

## Task 4: Apply Column Mask to Table

Apply the `mask_email` function to the `email` column of the `customers` table.

Hint: `ALTER TABLE ... ALTER COLUMN ... SET MASK ...`

**Guidance — Task 4**

Apply the mask function to a specific column using `ALTER TABLE ... ALTER COLUMN ... SET MASK`.

**Syntax:**
```sql
-- Apply mask
ALTER TABLE catalog.schema.customers
ALTER COLUMN email SET MASK catalog.schema.mask_email;

-- Remove mask
ALTER TABLE catalog.schema.customers
ALTER COLUMN email DROP MASK;
```

The function is referenced by its **full three-part name** (catalog.schema.function).

After applying, run a `SELECT` on the `email` column. Admins see the full address; all other users see the masked version (e.g. `jo***@***.***`).

**Things to think about**
- Does the column mask modify the underlying data stored in Delta? Or does it only affect how the value is returned to the caller?
- What happens if an admin user queries the `email` column?

In [0]:
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.customers
    ALTER COLUMN email SET MASK {CATALOG}.{SILVER_SCHEMA}.mask_email
""")
print("Column mask applied to email column")

In [0]:
# Verification -- query the table
masked_df = spark.sql(f"SELECT customer_id, email FROM {CATALOG}.{SILVER_SCHEMA}.customers LIMIT 5")
masked_df.display()
print("Task 4 PASSED -- check if email is masked for non-admin users")

## Task 5: Create a Row Filter Function

Create a function that restricts visibility of orders by `store_region`. Only users in the matching group can see rows for that region. Admins see all.

Hint: Use `is_account_group_member()` with multiple OR conditions.

**Guidance — Task 5**

A **Row Filter** is a SQL function that returns `BOOLEAN`. Rows where the function returns `FALSE` are invisible to the caller — they are filtered out before results are returned.

**Syntax:**
```sql
CREATE OR REPLACE FUNCTION catalog.schema.region_filter(region STRING)
RETURNS BOOLEAN
RETURN (
    is_account_group_member('admins')                          -- admins see all rows
    OR (is_account_group_member('us_team') AND region = 'East')
    OR (is_account_group_member('eu_team') AND region = 'West')
)
```

Key points:
- The **parameter name** (`region`) must match the column name used in `ON (column)` in Task 6
- The actual `store_region` values in this dataset are `'East'`, `'West'`, `'Central'`
- Users not in any of the listed groups receive **0 rows** (the function returns FALSE for all rows)
- Admins bypass filtering because the first condition always returns TRUE

**Things to think about**
- A user in both `us_team` and `eu_team` — which rows do they see?
- What happens to a user who belongs to none of the listed groups?

In [0]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.region_filter(region STRING)
    RETURNS BOOLEAN
    RETURN (
        is_account_group_member('admins')
        OR (is_account_group_member('us_team') AND region = 'East')
        OR (is_account_group_member('eu_team') AND region = 'West')
    )
""")
print("Row filter function created")

In [ ]:
# -- Validation --
fn_df = spark.sql(f"""
    SELECT routine_name
    FROM {CATALOG}.INFORMATION_SCHEMA.ROUTINES
    WHERE routine_schema = '{SILVER_SCHEMA}'
    AND routine_name = 'region_filter'
""")
assert fn_df.count() > 0, "region_filter function not found — check your CREATE FUNCTION statement"
print(f"Task 5 OK: region_filter function created in {CATALOG}.{SILVER_SCHEMA}")

## Task 6: Apply Row Filter to Table

Apply the `region_filter` function to the `orders` table.

Hint: `ALTER TABLE ... SET ROW FILTER ... ON (column)`

**Guidance — Task 6**

Apply the row filter function to the table using `ALTER TABLE ... SET ROW FILTER`.

**Syntax:**
```sql
-- Apply filter
ALTER TABLE catalog.schema.orders
SET ROW FILTER catalog.schema.region_filter ON (store_region);

-- Remove filter
ALTER TABLE catalog.schema.orders
DROP ROW FILTER;
```

The column name in `ON (store_region)` is **passed as the argument** to the filter function. It must match the function's parameter type (STRING).

After applying, run a `SELECT COUNT(*)` on the orders table. The visible row count will depend on which group the current user belongs to.

**Things to think about**
- The column in `ON (...)` must be a real column in the table. What error would you get if it does not exist?
- Can you attach two different row filters to the same table simultaneously?

In [0]:
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.orders
    SET ROW FILTER {CATALOG}.{SILVER_SCHEMA}.region_filter ON (store_region)
""")
print("Row filter applied to orders table")

In [0]:
# Verification -- check row count (admins should see all rows)
filtered_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.{SILVER_SCHEMA}.orders").first()["cnt"]
print(f"Visible rows: {filtered_count}")
print("Task 6 PASSED")

## Task 7: Query Table Privileges

Use `INFORMATION_SCHEMA.TABLE_PRIVILEGES` to verify who has access to what.

**Guidance — Task 7**

`INFORMATION_SCHEMA.TABLE_PRIVILEGES` lists all privileges granted on tables within the catalog — useful for auditing and compliance checks.

**Syntax:**
```sql
SELECT grantor, grantee, table_schema, table_name, privilege_type
FROM my_catalog.INFORMATION_SCHEMA.TABLE_PRIVILEGES
ORDER BY grantee, table_name
```

Key columns:

| Column | Meaning |
|--------|---------|
| `grantor` | Who granted the privilege |
| `grantee` | Who received it (user or group) |
| `table_schema` | Schema the table belongs to |
| `privilege_type` | e.g. `SELECT`, `MODIFY`, `ALL PRIVILEGES` |

**Things to think about**
- Does `TABLE_PRIVILEGES` show schema-level grants (from Task 1) or only table-level grants?
- How would you filter to find all privileges held by the `analysts` group specifically?

In [0]:
privs_df = spark.sql(f"""
    SELECT grantor, grantee, table_schema, table_name, privilege_type
    FROM {CATALOG}.INFORMATION_SCHEMA.TABLE_PRIVILEGES
    ORDER BY grantee, table_name
""")
privs_df.display()

In [0]:
# Verification
assert privs_df.count() > 0, "No privileges found"
print(f"Found {privs_df.count()} privilege entries")
print("Task 7 PASSED")

## Task 8: Remove Mask and Filter (Cleanup)

Remove the column mask and row filter applied earlier.

Hint: `ALTER TABLE ... ALTER COLUMN ... DROP MASK` and `ALTER TABLE ... DROP ROW FILTER`

**Guidance — Task 8**

The column mask and the row filter are removed with two separate `ALTER TABLE` statements.

**Syntax:**
```sql
-- Remove column mask from customers.email
ALTER TABLE catalog.schema.customers
ALTER COLUMN email DROP MASK;

-- Remove row filter from orders
ALTER TABLE catalog.schema.orders
DROP ROW FILTER;
```

Important distinction:
- `DROP MASK` / `DROP ROW FILTER` removes the **binding** between the function and the table
- The **mask/filter function itself remains in the catalog** and can be re-applied at any time
- After removal, all users immediately see the full email value and all order rows

**Things to think about**
- After `DROP MASK`, can all users now see the full email column? What about the underlying Parquet files — were they ever changed?
- What is the difference between dropping the mask binding and dropping the mask function?

In [0]:
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.customers
    ALTER COLUMN email DROP MASK
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.orders
    DROP ROW FILTER
""")

print("Cleanup complete -- mask and filter removed")

In [0]:
# Final verification
clean_df = spark.sql(f"SELECT customer_id, email FROM {CATALOG}.{SILVER_SCHEMA}.customers LIMIT 3")
clean_df.display()
print("Task 8 PASSED -- all governance controls removed")

## Task 9: DENY Privilege

DENY blocks access explicitly — overrides any GRANT on the same object.

**Scenario:** The `contractors` principal was mistakenly GRANTed `SELECT` on the Bronze schema.
Use DENY to block access to the `orders` table specifically.

**Expected syntax:**
```sql
DENY SELECT ON TABLE <catalog>.<schema>.<table> TO `<principal>`
```

**Guidance — Task 9**

`DENY` explicitly blocks access. It **overrides any GRANT** on the same object — even grants inherited from schema or catalog level.

**Syntax:**
```sql
DENY SELECT ON TABLE catalog.schema.orders TO `contractors`;
```

**GRANT vs REVOKE vs DENY:**

| Statement | Effect | Overrides GRANT? |
|-----------|--------|-----------------|
| `GRANT` | Allows access | — |
| `REVOKE` | Removes a previously issued GRANT | — |
| `DENY` | Explicitly blocks access | Yes — overrides all GRANTs |

Example: if `contractors` has `GRANT SELECT ON SCHEMA bronze`, a `DENY SELECT ON TABLE bronze.orders` blocks access to that specific table while keeping access to all other tables in the schema.

> The code is commented out because `contractors` must already exist as a group in the Databricks workspace. Uncomment to test in an environment where the group exists.

**Things to think about**
- A user is in both `analysts` (GRANT SELECT on schema) and `contractors` (DENY SELECT on table). Can they query the table?
- How is `DENY` different from `REVOKE`?

In [0]:
spark.sql(f"""
    DENY SELECT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders TO `contractors`
""")

spark.sql(f"SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders").display()

print("Task 9: DENY takes precedence over GRANT. DENY applied at TABLE level.")

In [0]:
# Verification: SHOW GRANTS includes both GRANT and DENY entries
grants = spark.sql(f"""SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders""") \
              .filter("privilege_type LIKE '%DENY%' OR privilege_type = 'SELECT'")
grants.display()
print("Task 9 complete: DENY verified via SHOW GRANTS")

## Summary

| Task | Concept | Key SQL |
|------|---------|--------|
| 1 | Privileges | `GRANT SELECT ON SCHEMA ... TO ...` |
| 2 | Metadata | `INFORMATION_SCHEMA.TABLES` |
| 3-4 | Column Mask | `ALTER TABLE ... ALTER COLUMN ... SET MASK fn` |
| 5-6 | Row Filter | `ALTER TABLE ... SET ROW FILTER fn ON (col)` |
| 7 | Audit | `INFORMATION_SCHEMA.TABLE_PRIVILEGES` |
| 8 | Cleanup | `DROP MASK`, `DROP ROW FILTER` |
| 9 | DENY | `DENY SELECT ON TABLE ... TO principal` |

← [10 — Governance & Security](../demo/10_governance_and_security.ipynb) | **[ README](../../../README.md)** | [11 — Exam Preparation →](../demo/11_exam_preparation.ipynb)